# 04 · Reorder behaviour: what do customers buy again?

**Question:** which products and departments are bought again and again? For a grocer that buys stock from known orders, repeat products are the most predictable demand: buying the right quantity of them reduces waste, especially for fresh food.

**Note:** `reordered = 1` if the customer had already bought the product in a previous order. In a customer's first order every product is new by definition, so **first orders are excluded** from reorder rates.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect("../data/instacart.duckdb", read_only=True)

def q(sql):
    """Run a SQL query on the DuckDB database and return a pandas DataFrame."""
    return con.sql(sql).df()

## 4.1 Overall reorder rate

In [ ]:
q("""
SELECT ROUND(AVG(reordered) * 100, 1) AS reorder_rate_all_orders
FROM order_products_prior
""")

In [ ]:
q("""
SELECT ROUND(AVG(op.reordered) * 100, 1) AS reorder_rate_excl_first
FROM order_products_prior op
JOIN orders o ON op.order_id = o.order_id
WHERE o.order_number > 1
""")

59% on all orders vs **63% excluding first orders**, where `reordered` is always 0. From the second order on, almost 2 products out of 3 are repeat purchases.

## 4.2 Reorder rate by department

In [ ]:
dept = q("""
SELECT d.department,
       COUNT(*) AS n_items,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_items,
       ROUND(AVG(op.reordered) * 100, 1)                  AS reorder_rate
FROM order_products_prior op
JOIN orders o      ON op.order_id = o.order_id
JOIN products p    ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
WHERE o.order_number > 1
GROUP BY d.department
ORDER BY reorder_rate DESC
""")
dept

In [ ]:
d = dept.sort_values("reorder_rate")
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(d["department"], d["reorder_rate"], color="#2a78d6", height=0.7)
for i, v in enumerate(d["reorder_rate"]):
    ax.text(v + 0.8, i, f"{v:.0f}%", va="center", fontsize=9, color="#444")
ax.xaxis.set_visible(False)
for side in ["top", "right", "bottom"]:
    ax.spines[side].set_visible(False)
ax.spines["left"].set_color("#ccc")
ax.tick_params(axis="y", labelsize=9, length=0)
ax.set_title("Reorder rate by department (excluding first orders)", loc="left", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/reorder_by_department.png", dpi=150, bbox_inches="tight")
plt.show()

- **Fresh departments lead:** dairy & eggs 71.4%, produce 69.3%, bakery 67.1%, deli 65.1%. Produce is also the largest department (29.3% of items).
- **Pantry, personal care and household are reordered least** (34–43%): long-lasting goods bought occasionally.
- Meat & seafood (60.8%) is lower than other fresh departments, possibly because customers rotate between products week to week (hypothesis).
- `missing` and `other` are departments without a real category (<0.5% of items): they do not change the conclusions.

## 4.3 Top 20 products

In [ ]:
q("""
SELECT p.product_name,
       COUNT(*)                          AS n_orders,
       ROUND(AVG(op.reordered) * 100, 1) AS reorder_rate
FROM order_products_prior op
JOIN orders o   ON op.order_id = o.order_id
JOIN products p ON op.product_id = p.product_id
WHERE o.order_number > 1
GROUP BY p.product_id, p.product_name
ORDER BY n_orders DESC
LIMIT 20
""")

All 20 are fresh (19 fruit & vegetables, 1 milk), 15 are organic, and all are reordered 68–90% of the time, above the 63% average. Bananas alone account for 800k+ purchases. The most perishable products are also the most predictable ones.

## 4.4 Fresh vs other products

In [ ]:
q("""
SELECT
    CASE
        WHEN d.department IN ('produce', 'dairy eggs', 'meat seafood', 'bakery', 'deli') THEN 'Fresh'
        ELSE 'Other'
    END AS category,
    COUNT(*)                                           AS n_items,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_items,
    ROUND(AVG(op.reordered) * 100, 1)                  AS reorder_rate
FROM order_products_prior op
JOIN orders o      ON op.order_id = o.order_id
JOIN products p    ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
WHERE o.order_number > 1
GROUP BY category
ORDER BY category
""")

Fresh departments are **55% of all items** (consistent with the department table: 29.3 + 16.7 + 3.6 + 3.2 + 2.2) and are reordered far more often than other products: **69.2% vs 55.4%**.

## 4.5 Cart position of reordered vs new products

In [ ]:
q("""
SELECT op.reordered,
       COUNT(*)                            AS n_items,
       ROUND(AVG(op.add_to_cart_order), 1) AS avg_cart_position
FROM order_products_prior op
JOIN orders o ON op.order_id = o.order_id
WHERE o.order_number > 1
GROUP BY op.reordered
ORDER BY op.reordered
""")

Reordered products are added to the cart earlier (average position **7.6 vs 9.7**): customers put their usual items in first and then look for new ones. Consistency check: 19,126,536 reordered items out of 30,356,421 = 63.0%, the same rate as in 4.1. Part of the gap may be driven by basket size.

## Findings

1. **Most of the basket is repeat purchases.** From the second order on, 63% of products are items the customer had already bought (59% if first orders are included).
2. **Fresh food is the biggest and the most predictable part of the basket.** Fresh departments (produce, dairy & eggs, meat & seafood, bakery, deli) are 55% of all items and are reordered much more often than other products (69.2% vs 55.4%). Dairy & eggs (71.4%) and produce (69.3%) lead the ranking; pantry, household and personal care are reordered least (34–43%).
3. **The top sellers are all fresh, and customers add them first.** The 20 most ordered products are all fresh (19 fruit & vegetables, 1 milk; 15 organic) with reorder rates of 68–90%. Reordered products are added to the cart earlier than new ones (average position 7.6 vs 9.7).

**Recommendation:** for a grocer that knows orders in advance, the most perishable products are also the most predictable: buying fresh stock from known repeat demand can cut waste where it costs most. In the app, a "Buy again" section at the top of the shopping flow would match how customers already shop and make the weekly order faster.

**Caveats:** no prices (items ≠ revenue); department labels are the retailer's own ("missing" and "other" are <0.5% of items); the cart-position gap may be partly driven by basket size.